# Fase 4 — Inferência em Vídeo

Concatena os clipes de vídeo do cenário e roda os dois modelos treinados sobre o resultado, salvando as saídas anotadas.

**Antes de rodar:** faça upload dos 3 clipes originais (ver `video/input/README.md` no repositório para as fontes no Pexels) para `PROJECT_DIR/video/input/` no Google Drive.

## Setup

In [ ]:
!pip install -q ultralytics opencv-python pandas pyarrow matplotlib pillow kaggle

from pathlib import Path

# Armazenamento persistente compartilhado entre os notebooks: monta o Google
# Drive e usa uma pasta fixa. Troque o caminho se preferir outra estrutura.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/vc-seguranca-trabalho')
except ImportError:
    # Execucao fora do Colab (teste local) - usa uma pasta local.
    PROJECT_DIR = Path('./vc-seguranca-trabalho').resolve()

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
ROOT = PROJECT_DIR
print("Diretorio do projeto:", ROOT)


## 10. Concatenar o vídeo final

Concatena os 3 clipes em um único vídeo ≥30s, padronizando resolução (1280×720, letterbox para os clipes em retrato) e taxa de quadros (25fps).

In [ ]:
import cv2
import numpy as np

INPUT_DIR = ROOT / "video" / "input"
OUTPUT_PATH = INPUT_DIR / "video_final_canteiro_obra.mp4"

# Ordem de concatenacao: paisagem primeiro (mais parecido com o dataset de
# treino), depois os dois clipes em retrato. Fontes/licenca em
# video/input/README.md (Pexels License).
SOURCE_ORDER = [
    "19832492-hd_1280_720_25fps.mp4",
    "14626383_720_1280_30fps.mp4",
    "15518317_720_1280_60fps.mp4",
]

TARGET_W, TARGET_H = 1280, 720
TARGET_FPS = 25.0


def letterbox(frame, target_w, target_h):
    h, w = frame.shape[:2]
    scale = min(target_w / w, target_h / h)
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)
    x_off = (target_w - new_w) // 2
    y_off = (target_h - new_h) // 2
    canvas[y_off : y_off + new_h, x_off : x_off + new_w] = resized
    return canvas


missing = [f for f in SOURCE_ORDER if not (INPUT_DIR / f).exists()]
if missing:
    print("Faca upload destes clipes para", INPUT_DIR, "antes de continuar:")
    for f in missing:
        print(" -", f)
    print("Fontes: video/input/README.md no repositorio (Pexels).")
else:
    writer = cv2.VideoWriter(
        str(OUTPUT_PATH), cv2.VideoWriter_fourcc(*"mp4v"), TARGET_FPS, (TARGET_W, TARGET_H),
    )

    total_frames_written = 0
    for filename in SOURCE_ORDER:
        src_path = INPUT_DIR / filename
        cap = cv2.VideoCapture(str(src_path))
        src_fps = cap.get(cv2.CAP_PROP_FPS) or TARGET_FPS
        src_frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        duration = src_frame_count / src_fps
        n_target_frames = int(duration * TARGET_FPS)
        src_indices = [int(i * src_fps / TARGET_FPS) for i in range(n_target_frames)]

        frames_written_this_clip = 0
        idx_to_frame = {}
        current_idx = -1
        for target_idx in src_indices:
            if target_idx not in idx_to_frame:
                while current_idx < target_idx:
                    ok, frame = cap.read()
                    current_idx += 1
                    if not ok:
                        break
                if ok:
                    idx_to_frame[target_idx] = letterbox(frame, TARGET_W, TARGET_H)
            if target_idx in idx_to_frame:
                writer.write(idx_to_frame[target_idx])
                frames_written_this_clip += 1

        cap.release()
        total_frames_written += frames_written_this_clip
        print(f"{filename}: {frames_written_this_clip} frames escritos "
              f"({frames_written_this_clip / TARGET_FPS:.1f}s)")

    writer.release()
    total_duration = total_frames_written / TARGET_FPS
    print(f"\nVideo final: {OUTPUT_PATH}")
    print(f"Duracao total: {total_duration:.1f}s ({total_frames_written} frames a {TARGET_FPS}fps)")


## 13. Inferência em vídeo

Roda o detector e o segmentador sobre o vídeo final, salvando as saídas anotadas em `video/output/`.

In [ ]:
from ultralytics import YOLO

VIDEO_INPUT = ROOT / "video" / "input" / "video_final_canteiro_obra.mp4"
VIDEO_OUTPUT_DIR = ROOT / "video" / "output"

DETECTION_WEIGHTS = ROOT / "models" / "detection" / "css_yolov8n_baseline" / "weights" / "best.pt"
SEGMENTATION_WEIGHTS = ROOT / "models" / "segmentation" / "coco_person_yolov8n_seg_baseline" / "weights" / "best.pt"

CONF_THRESHOLD = 0.25


def run_inference(weights_path, run_name):
    model = YOLO(str(weights_path))
    model.predict(
        source=str(VIDEO_INPUT), conf=CONF_THRESHOLD, save=True,
        project=str(VIDEO_OUTPUT_DIR), name=run_name, exist_ok=True, verbose=False,
    )
    print(f"Inferencia concluida: {run_name}")


print("=== Inferencia do detector (EPI) no video ===")
run_inference(DETECTION_WEIGHTS, "deteccao_epi")

print("\n=== Inferencia do segmentador (pessoas) no video ===")
run_inference(SEGMENTATION_WEIGHTS, "segmentacao_pessoas")


## Resultado

Os vídeos anotados ficam em `PROJECT_DIR/video/output/deteccao_epi/` e `.../segmentacao_pessoas/` — baixe do Drive para assistir ou usar no vídeo-pitch.